# 지수 방법론 분석 노트북 (v2)
## 국내 지수 100개 방법론 분석 및 2026년 6월 리밸런싱 현황

**분석 대상**: Google Drive 폴더 내 KRX, FnGuide, iSelect, KEDI, MSCI 등 100개 지수 방법론 PDF  
**현재 날짜**: 2026년 6월 8일  
**오프라인 실행 가능**: 모든 데이터가 이 노트북에 내장되어 있습니다

### v2 주요 변경사항
- KRX 8단계 Rule Engine 파라미터 추가 (cap_threshold, liq_pct, keep_buffer, new_buffer, ...)
- provider_type 분류 추가 (KRX, FnGuide계열, iSelect, KEDI, MSCI, 기타)
- 상세 유니버스 구성 필드 추가 (float_rate_min, listing_period_months, excluded_types 등)

## 1. 환경 설정

In [ ]:
# 필요 패키지 (openpyxl은 Excel 생성 시 필요)
# pip install openpyxl

import json
import os
from pathlib import Path
from collections import defaultdict, Counter

print('환경 설정 완료')

## 2. 분석 데이터 (100개 지수 방법론 v2)

PDF 100개를 v2 스키마로 분석한 결과가 아래 셀에 내장되어 있습니다.  
**오프라인 환경에서도 바로 실행 가능합니다.**

### v2 스키마 주요 필드
| 필드 | 설명 |
|------|------|
| `provider_type` | KRX / FnGuide계열 / iSelect / KEDI / MSCI / 기타 |
| `target_n` | 지수 구성종목 목표 수 |
| `cap_threshold` | KRX 1차 선정 시총 누적비중(%) |
| `liq_pct` | KRX 1차 선정 거래대금 누적비중(%) |
| `keep_buffer` | KRX 2차 선정 편입유지 버퍼(%) |
| `new_buffer` | KRX 2차 선정 신규편입 버퍼(%) |
| `tertiary_mode` | KRX 3차 선정 방식 (global/conservative) |
| `sector_count` | 섹터 수 (KRX 섹터 지수) |
| `ceiling_pct` | 종목별 비중 상한(%) |

In [ ]:
# 100개 지수 방법론 분석 결과 (v2 스키마, PDF에서 추출)
# Google Drive: https://drive.google.com/drive/folders/1ewDDJ-1pa2Ii_rAP1FrFWCuC_AKfkWZJ

ALL_INDEX_DATA = [
  {
    "file": "10_KRX_KO200_IT_Sector.pdf",
    "index_name_ko": "KOSPI 200 정보기술",
    "index_name_en": "KOSPI 200 Information Technology",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 6,
    "review_date_rule": "정기변경월의 전전월 말일",
    "effective_date_rule": "6월/12월 KOSPI200 선물 최종거래일 익일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 6,
    "excluded_types": "관리종목,정리매매,부동산투자회사,선박투자회사,사회기반시설투융자회사,기업인수목적회사",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "상장폐지결정,관리종목지정,투자주의환기종목지정",
    "june_2026_rebalancing": "Y",
    "sector_count": 1,
    "sector_map_type": "KOSPI10",
    "sector_min_pct": null,
    "cap_threshold": 85,
    "liq_pct": 85,
    "keep_buffer": 110,
    "new_buffer": 90,
    "tertiary_mode": "global",
    "small_excl_rank": null,
    "large_cap_special_rank": 50,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "11_FG_SecondaryBattery.pdf",
    "index_name_ko": "FnGuide 2차전지 산업 지수",
    "index_name_en": "FnGuide Secondary Battery Industry Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 25,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "2,5,8,11월 말 마지막 영업일",
    "effective_date_rule": "3,6,9,12월 선물옵션 만기일 익주 첫~세 번째 영업일",
    "rebalancing_window_days": 3,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목,투자주의환기종목,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "신규상장,물적분할",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "스코어 상위25종목"
  },
  {
    "file": "12_FG_Korea_TOP10.pdf",
    "index_name_ko": "FnGuide Korea TOP10 지수",
    "index_name_en": "FnGuide Korea TOP10 Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "2,8월 마지막 영업일",
    "effective_date_rule": "3,9월 선물옵션 만기일(D) 이후 D+2",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목,선박투자회사,REITs,인프라투자회사,ETF,ETN,해외주식",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "유동시총 상위10종목"
  },
  {
    "file": "13_FG_Dividend.pdf",
    "index_name_ko": "FnGuide 고배당주 지수",
    "index_name_en": "FnGuide High Dividend Stocks Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "개편 직전 종목선정일",
    "effective_date_rule": "6,12월 선물옵션만기일 포함 5영업일째(T+4)",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "선박투자회사,REITs,ETF,ETN,관리종목,투자유의종목",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": 10,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "시총 상위200종목 내 배당수익률 상위30"
  },
  {
    "file": "14_FG_Ship_TOP3.pdf",
    "index_name_ko": "FnGuide 조선 TOP3 플러스 지수",
    "index_name_en": "FnGuide Shipbuilding TOP3 Plus Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 13,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "1,4,7,10월 마지막 영업일",
    "effective_date_rule": "2,5,8,11월 선물옵션 만기일 D+2",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목,투자주의환기종목,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC",
    "ceiling_pct": 25,
    "ceiling_effective_days": 3,
    "adhoc_trigger": "3영업일 연속 30% 초과 시 D+3에 조정",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "조선TOP3+플러스유니버스 시총상위10"
  },
  {
    "file": "15_FG_Korea_TOP5.pdf",
    "index_name_ko": "FnGuide TOP 5 Plus 지수",
    "index_name_en": "FnGuide TOP 5 Plus Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "5월 말 마지막 영업일",
    "effective_date_rule": "6월 선물옵션 만기일 익주 첫 번째 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목,투자주의환기종목,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC,외국인투자제한",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "유동시총 상위30 내 배당+시총 스코어 상위10"
  },
  {
    "file": "16_KRX_KO200_CC5.pdf",
    "index_name_ko": "코스피 200 커버드콜 지수",
    "index_name_en": "KOSPI 200 Covered Call Index",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "옵션전략",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": null,
    "effective_date_rule": null,
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "17_FG_K_defense.pdf",
    "index_name_ko": "FnGuide K-방위산업 지수",
    "index_name_en": "FnGuide K-Defense Industry Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "5,11월 마지막 영업일",
    "effective_date_rule": "6,12월 선물옵션 만기일 D+2",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목,투자주의환기종목,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC",
    "ceiling_pct": 20,
    "ceiling_effective_days": 2,
    "adhoc_trigger": "매월말 30% 초과 시 다음월 D+2에 25%로 조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "방산유사도 상위30 → 팩터 상위10"
  },
  {
    "file": "18_FG_REITs.pdf",
    "index_name_ko": "FnGuide 리츠부동산인프라 지수",
    "index_name_en": "FnGuide REITs Real Estate Infrastructure Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI",
    "weight_method": "시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "5,11월 말 마지막 영업일",
    "effective_date_rule": "6,12월 선물옵션 만기일 D+2~D+6(5영업일)",
    "rebalancing_window_days": 5,
    "float_rate_min": 0,
    "listing_period_months": null,
    "excluded_types": "관리종목,투자주의환기종목",
    "ceiling_pct": 17,
    "ceiling_effective_days": null,
    "adhoc_trigger": "신규 REITs/인프라 상장 시 D+6 이후 편입",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "REITs/인프라 배당수익률 상위30"
  },
  {
    "file": "19_ISELECT_K_ROBOT.pdf",
    "index_name_ko": "iSelect K-로봇테마 지수",
    "index_name_en": "iSelect K-Robot Theme Index",
    "provider": "iSelect(NH투자증권)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일",
    "effective_date_rule": "6,12월 KOSPI200 선물옵션 만기일 2영업일 이후",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리대상종목,정리매매종목,선박투자회사,REITs,인프라투자회사,ETF,ETN,SPAC",
    "ceiling_pct": 8,
    "ceiling_effective_days": null,
    "adhoc_trigger": "상장폐지,관리종목지정,기업분할,합병",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "AI키워드스코어 상위30종목"
  },
  {
    "file": "1_KRX_KOSPI200.pdf",
    "index_name_ko": "KOSPI 200",
    "index_name_en": "KOSPI 200",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 200,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 6,
    "review_date_rule": "정기변경월의 전전월 말일",
    "effective_date_rule": "6월/12월 KOSPI200 선물 최종거래일 익일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 6,
    "excluded_types": "관리종목,정리매매,부동산투자회사,선박투자회사,사회기반시설투융자회사,기업인수목적회사",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "상장폐지결정,관리종목지정,신규상장(시총50위이내 특례)",
    "june_2026_rebalancing": "Y",
    "sector_count": 10,
    "sector_map_type": "KOSPI10",
    "sector_min_pct": "N",
    "cap_threshold": 85,
    "liq_pct": 85,
    "keep_buffer": 110,
    "new_buffer": 90,
    "tertiary_mode": "global",
    "small_excl_rank": null,
    "large_cap_special_rank": 50,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "20_FG_AISEMI_TOP3.pdf",
    "index_name_ko": "FnGuide AI 반도체 TOP3+ 지수",
    "index_name_en": "FnGuide AI Semiconductor TOP3 Plus Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 20,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "고정비중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "3,6,9,12월 마지막 영업일",
    "effective_date_rule": "3,6,9,12월 선물옵션 만기일 D+2",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목,투자주의환기종목,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC,지주회사",
    "ceiling_pct": 25,
    "ceiling_effective_days": 2,
    "adhoc_trigger": "5영업일 연속 30% 초과 시 D+2에 25%로 조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "FICS반도체+전자장비 내 매출필터 후 재무스코어 상위20"
  },
  {
    "file": "21_KEDI_AI_power_TOP3.pdf",
    "index_name_ko": "KEDI 코리아AI전력기기TOP3플러스 지수",
    "index_name_en": "KEDI Korea AI Electrical Power Equipment TOP3 Plus Index",
    "provider": "KEDI(한국경제신문지수)",
    "provider_type": "KEDI",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "고정비중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "6,12월 두 번째 목요일(선물옵션 만기일, D)",
    "effective_date_rule": "D+3 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리종목,투자유의종목",
    "ceiling_pct": 25,
    "ceiling_effective_days": 3,
    "adhoc_trigger": "3영업일 연속 30% 초과 시 T+3에 조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "KICS업종+전력기기유사도점수 상위10"
  },
  {
    "file": "22_WISE_SecondaryBattery.pdf",
    "index_name_ko": "WISE 2차전지 테마 지수",
    "index_name_en": "WISE Secondary Cell Theme Index",
    "provider": "FnGuide(WISE)",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "시총가중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "3,6,9,12월 마지막 영업일",
    "effective_date_rule": "1,4,7,10월 옵션 만기일 익일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "선박,부동산투자회사,ETF,ETN,관리종목,투자유의종목",
    "ceiling_pct": 15,
    "ceiling_effective_days": null,
    "adhoc_trigger": "기업합병,기업분할,관리종목지정,화의신청,신규상장",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "2차전지 키워드 상위5위+매출기준"
  },
  {
    "file": "23_WISE_SSE_Bond.pdf",
    "index_name_ko": "Wise 삼성전자 채권혼합 지수",
    "index_name_en": "Wise Samsung Electronics Balanced Index",
    "provider": "FnGuide(WISE)",
    "provider_type": "FnGuide계열",
    "index_type": "채권혼합",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "고정비중",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "매일 3:7 비중 유지(Constant Mix)",
    "effective_date_rule": "매일 재조정",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "24_KRX_KOSPI100.pdf",
    "index_name_ko": "KOSPI 100",
    "index_name_en": "KOSPI 100",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 100,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 6,
    "review_date_rule": "KOSPI200 정기심사 기준일과 동일",
    "effective_date_rule": "6월/12월 KOSPI200 선물 최종거래일 익일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 6,
    "excluded_types": "관리종목,정리매매,부동산투자회사,선박투자회사,사회기반시설투융자회사,기업인수목적회사",
    "ceiling_pct": 30,
    "ceiling_effective_days": null,
    "adhoc_trigger": "KOSPI200 수시변경으로 제외되는 종목",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "KOSPI10",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": 120,
    "new_buffer": 80,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "25_FG_AISEMI_soboojang.pdf",
    "index_name_ko": "FnGuide AI 반도체 소부장 지수",
    "index_name_en": "FnGuide AI Semiconductor Materials & Equipment Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 20,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "5,11월 마지막 영업일",
    "effective_date_rule": "6,12월 선물옵션 만기일 D+2",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목,투자주의환기종목,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC",
    "ceiling_pct": 20,
    "ceiling_effective_days": 3,
    "adhoc_trigger": "매월말 30% 초과 시 D+3에 25%로 조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "FICS내 키워드유사도 상위50→시총상위20"
  },
  {
    "file": "26_KRX_SecondaryBattery_TOP10.pdf",
    "index_name_ko": "KRX 2차전지 TOP 10 지수",
    "index_name_en": "KRX Secondary Battery TOP 10 Index",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "1,7월 최종 매매거래일",
    "effective_date_rule": "3,9월 KOSPI200 선물 최종거래일 익일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목,정리매매,투자주의환기종목,집합투자기구,외국주권,부동산투자회사,선박투자회사,사회기반시설투융자회사,기업인수목적회사",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "상장폐지결정,관리종목지정,투자주의환기종목지정",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": 80,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "27_MKF_Hyundai.pdf",
    "index_name_ko": "MKF 현대차그룹 지수",
    "index_name_en": "MKF Hyundai Motor Group Index",
    "provider": "FnGuide(MKF)",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 기준",
    "effective_date_rule": "6,12월 KOSPI200 선물 최종거래일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리종목,투자유의종목,정리매매",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "현대차그룹 계열회사 전체"
  },
  {
    "file": "28_KRX_KO200_WC_ATM.pdf",
    "index_name_ko": "코스피 200 위클리 커버드콜 ATM 지수",
    "index_name_en": "KOSPI 200 Weekly Covered Call ATM Index",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "옵션전략",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": null,
    "effective_date_rule": null,
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "2_FG_SEMI_TOP10.pdf",
    "index_name_ko": "FnGuide 반도체 TOP10 지수",
    "index_name_en": "FnGuide Semiconductor TOP10 Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": 1,
    "review_date_rule": "매년 3월, 9월 말 마지막 영업일",
    "effective_date_rule": "매년 4월, 10월 선물옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC, 유동비율10%미만",
    "ceiling_pct": 30,
    "ceiling_effective_days": 5,
    "adhoc_trigger": "매 월말 영업일 기준 T-4~T까지 5일 연속 특정 종목 비중 30% 초과 시 T+3에 비중 재계산 반영",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "FICS 중분류 '반도체' 종목 중 종목선정일 기준 최근 1개월 평균 단순시가총액 상위 10종목"
  },
  {
    "file": "30_KRX_SECU.pdf",
    "index_name_ko": "KRX 증권 지수",
    "index_name_en": "KRX Securities Sector Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일(심사기준일) 기준 KRX 중대형 TMI 구성종목",
    "effective_date_rule": "매년 1회, KOSPI200 선물시장 9월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "부동산투자회사, 사회기반시설투융자회사",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "KRX TMI 수시변경 방법론 동일 적용 (부적합종목 제외, 신규상장, 합병, 기업분할)",
    "june_2026_rebalancing": "N",
    "sector_count": 17,
    "sector_map_type": "KRX통합",
    "sector_min_pct": "N",
    "cap_threshold": 95,
    "liq_pct": 90,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "31_WISE_SE_SKH_BOND.pdf",
    "index_name_ko": "Wise 삼성전자 SK하이닉스 채권혼합 지수",
    "index_name_en": "Wise Samsung Electronics SK Hynix Bond Balanced Index",
    "provider": "FnGuide(Wise)",
    "provider_type": "FnGuide계열",
    "index_type": "채권혼합",
    "target_n": 2,
    "market_scope": "KOSPI",
    "weight_method": "고정비중",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "해당없음(매일 Constant Mix 방식으로 25:25:50 조정)",
    "effective_date_rule": "해당없음(매일 자동 리밸런싱)",
    "rebalancing_window_days": 0,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "삼성전자·SK하이닉스 종목 변동 이벤트 발생 시 지수위원회 검토",
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "삼성전자 25%, SK하이닉스 25%, KAP 국고채 Focus 지수 50% 고정비중"
  },
  {
    "file": "32_ISELECT_NUC.pdf",
    "index_name_ko": "iSelect 원자력 지수",
    "index_name_en": "iSelect Nuclear Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 15,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일(심사일) 기준",
    "effective_date_rule": "매년 6월, 12월 코스피200 선물만기일 4영업일 이후",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리대상종목, 정리매매종목, 투자주의환기종목, 상장폐지확정종목, 선박투자, 인프라투자, 해외주, REITs, ETN&ETF, SPAC, 유동주식비율10%미만, 1년간 상장폐지실질심사대상",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "전월 마지막 영업일 기준 비중 30% 초과 종목이 당월 선물옵션 만기일 종가 기준으로도 30% 초과 시 T+4에 25%로 재조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "AI 기반 키워드 스코어링 상위 40위 중 시가총액 상위 15종목"
  },
  {
    "file": "33_ISELECT_KDEF_SPA.pdf",
    "index_name_ko": "iSelect K방산&우주 지수",
    "index_name_en": "iSelect K Defense & Space Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 15,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일(심사일) 기준 20영업일 평균",
    "effective_date_rule": "매년 1월, 4월, 7월, 10월 옵션만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC, 1년간 상장폐지실질심사대상",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "지수위원회(지수자문위원회 포함) 판단에 따른 수시 편출입",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "텍스트마이닝 키워드 스코어 상위 15종목 (시총 1500억 이상, 20영업일 평균 거래대금 5억 이상)"
  },
  {
    "file": "34_FG_HY_Bank.pdf",
    "index_name_ko": "FnGuide 은행 고배당 플러스 TOP 10 지수",
    "index_name_en": "FnGuide Bank High Dividend Plus TOP 10 Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매년 5월, 11월 마지막 영업일",
    "effective_date_rule": "매년 6월, 12월 선물옵션 만기일(D) 기준 2영업일째(D+2)",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 유동주식비율10%미만, 3개월 평균 유동시가총액 5000억 미만",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "재무 데이터 미반영 회계손실, 중대한 편출사유, 영업손실 확대 시 지수위원회 검토",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "FICS Industry '상업은행' 종목 중 3년 연속 현금 배당, 예상배당수익률 KOSPI 대비 높은 종목 중 시가총액 상위 10종목"
  },
  {
    "file": "35_Akros_AI_Power.pdf",
    "index_name_ko": "KEDI 미국 AI 전력 인프라 지수",
    "index_name_en": "KEDI US AI Electric Power Infrastructure Index",
    "provider": "KEDI(Akros Technologies)",
    "provider_type": "KEDI",
    "index_type": "주식",
    "target_n": 20,
    "market_scope": "해외",
    "weight_method": "기타",
    "frequency": "분기4회",
    "review_period_months": 3,
    "review_date_rule": "매년 3월, 6월, 9월, 12월 마지막 거래일(D)",
    "effective_date_rule": "Determination Date(D) + 3 거래일",
    "rebalancing_window_days": 3,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "NYSE/NYSE American/NASDAQ 미상장 종목, 유동시총 50억달러 미만(원전 관련 KAICS 221113 제외), 3개월 일평균 거래대금 100만달러 미만",
    "ceiling_pct": 10,
    "ceiling_effective_days": null,
    "adhoc_trigger": "신규 상장 종목: 상장일 이후 최초 미국 옵션 만기일에 편입 가능",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "NEXUS 키워드('AI Electricity', 'Nuclear Power') 스코어 상위 10종목씩 총 20종목, 상위 2종목 각 10% 고정, 나머지 18종목 유동시총가중(최대7%, 최소3%)"
  },
  {
    "file": "36_ISELECT_K_NUC.pdf",
    "index_name_ko": "iSelect 코리아 원자력 지수",
    "index_name_en": "iSelect Korea Nuclear Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 20,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일(심사일) 기준 20영업일 평균",
    "effective_date_rule": "매년 1월, 4월, 7월, 10월 옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 우선주, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC, 1년간 상장폐지실질심사대상",
    "ceiling_pct": 25,
    "ceiling_effective_days": 5,
    "adhoc_trigger": "종가 비중 30% 초과 5영업일 유지 시 익일에 25%로 조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "NLP 키워드 스코어 상위 30개 중 투자적정성 충족 후 원자력 유관 매출 발생 기업 최소 10~최대 20종목"
  },
  {
    "file": "37_FG_SecondaryBattery_SBJ.pdf",
    "index_name_ko": "FnGuide 2차전지소재 지수",
    "index_name_en": "FnGuide Secondary Battery Material Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "분기4회",
    "review_period_months": 3,
    "review_date_rule": "매년 3, 6, 9, 12월 마지막 영업일",
    "effective_date_rule": "매년 1, 4, 7, 10월 옵션 만기일(D) 기준 2영업일째(D+2)에 2영업일에 걸쳐 정기변경",
    "rebalancing_window_days": 2,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC, 유동비율10%미만, 시가총액2000억미만",
    "ceiling_pct": 20,
    "ceiling_effective_days": 5,
    "adhoc_trigger": "매 월말 영업일 기준 T-4~T까지 5일 연속 특정 종목 비중 30% 초과 시 T+3에 비중 재계산 반영",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "20영업일 평균 거래대금 전체시장 상위90% 및 시가총액 상위90% 충족 종목 중 키워드 유사도(TF-IDF) 상위 최대 30종목"
  },
  {
    "file": "38_ISELECT_SHIP_TOP10.pdf",
    "index_name_ko": "iSelect 조선TOP10 지수",
    "index_name_en": "iSelect Shipbuilding TOP10 Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일(심사일) 기준 20영업일 평균",
    "effective_date_rule": "매년 6월, 12월 옵션 만기일 익영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 우선주, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC, 1년간 상장폐지실질심사대상",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "매월 마지막 영업일 종가 기준 비중 30% 초과 종목 있는 경우 27%로 재조정, T+3에 시행",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "NLP 키워드 스코어 상위 25종목 중 직전 사업보고서 기준 선박건조 매출 50% 이상 종목 시총 상위 10종목"
  },
  {
    "file": "39_KEDI_MEGA_Tech.pdf",
    "index_name_ko": "KEDI 메가테크 지수",
    "index_name_en": "KEDI Mega Tech Index",
    "provider": "KEDI(한국경제신문)",
    "provider_type": "KEDI",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매년 5월말, 11월말 마지막 영업일",
    "effective_date_rule": "매년 6월, 12월 선물옵션 만기일 이후 2영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 선박투자회사, 인프라투자회사, 해외기업, 뮤추얼펀드, REITs, ETF, ETN, SPAC, 시가총액1000억미만, 유동비율10%미만, 60영업일평균거래대금10억미만",
    "ceiling_pct": 8,
    "ceiling_effective_days": null,
    "adhoc_trigger": "상장폐지, 관리종목 지정, 지수구성종목간 합병 등 이벤트 D+2에 차순위 종목 편입",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": 7,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "5개 메가테크 산업(설문조사 산업평가점수 상위 5개, 기존 편입 산업은 7위 이내 시 우선 선정) × 각 6종목"
  },
  {
    "file": "3_KRX_SEMI.pdf",
    "index_name_ko": "KRX 반도체 지수",
    "index_name_en": "KRX Semiconductor Sector Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일(심사기준일) 기준 KRX 중대형 TMI 구성종목",
    "effective_date_rule": "매년 1회, KOSPI200 선물시장 9월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "부동산투자회사, 사회기반시설투융자회사",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "KRX TMI 수시변경 방법론 동일 적용",
    "june_2026_rebalancing": "N",
    "sector_count": 17,
    "sector_map_type": "KRX통합",
    "sector_min_pct": "N",
    "cap_threshold": 95,
    "liq_pct": 90,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "40_KEDI_ROBOT.pdf",
    "index_name_ko": "KEDI 코리아휴머노이드로봇산업 지수",
    "index_name_en": "KEDI Korea Humanoid Robot Industry Index",
    "provider": "KEDI(한국경제신문)",
    "provider_type": "KEDI",
    "index_type": "주식",
    "target_n": 15,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "1,4,7,10월 옵션 만기일의 10영업일 전(D-10)",
    "effective_date_rule": "1,4,7,10월 옵션 만기일(D) 주식시장 종료 시점",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "비보통주, 시가총액 2000억 미만, 3개월 일평균 거래대금 10억 미만, KICS 분류 미해당 종목",
    "ceiling_pct": 15,
    "ceiling_effective_days": 5,
    "adhoc_trigger": "T-4~T까지 5영업일 연속 비중 30% 초과 시 25%로 축소, T+3 반영",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "LLM 기반 유사도 평가(유사도점수80%+시가총액20%) 편입점수 상위 최대 15종목"
  },
  {
    "file": "41_FG_Network_Infra.pdf",
    "index_name_ko": "FnGuide 네트워크 인프라 지수",
    "index_name_en": "FnGuide Network Infrastructure Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 60,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매년 5월, 11월 말 마지막 영업일",
    "effective_date_rule": "매년 6월, 12월 선물옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC, 유동비율10%미만, 60영업일 일평균 거래대금 2억 미만, 단순시가총액500억미만",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "재무 데이터 미반영 회계손실, 중대한 편출사유, 영업손실 확대 시 지수위원회 검토",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "TF-IDF 코사인 유사도 순위 80위 이내, FICS 대분류 'IT' 해당 종목, 단순시가총액 상위 60위 이내"
  },
  {
    "file": "42_KRX_Fi_HY_CC.pdf",
    "index_name_ko": "코스피 200 금융 고배당 TOP 10 타겟 15%분배 위클리 커버드콜 지수",
    "index_name_en": "KOSPI 200 Financial High Dividend TOP 10 Target 15% Distribution Weekly Covered Call Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "옵션전략",
    "target_n": 10,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "해당없음(커버드콜 전략 지수)",
    "effective_date_rule": "해당없음",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "해당없음",
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": "코스피 200 금융 고배당 TOP 10 타겟 커버리지 위클리 커버드콜 지수에서 연 15% 고정 분배금 월별 차감 산출"
  },
  {
    "file": "43_FG_HY_BOND.pdf",
    "index_name_ko": "FnGuide 고배당 채권혼합 지수",
    "index_name_en": "FnGuide High Dividend Balanced Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "채권혼합",
    "target_n": 30,
    "market_scope": "KOSPI",
    "weight_method": "고정비중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "FnGuide 고배당주 지수 개편일 기준(매년 6, 12월 선물옵션만기일 포함 5영업일째)",
    "effective_date_rule": "매년 6, 12월 선물옵션만기일 포함 5영업일째(T+4)",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "해당없음",
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "FnGuide 고배당주 지수 40% + MKF 국고채 3년 지수 60%"
  },
  {
    "file": "44_WISE_BIG_HY_TOP10.pdf",
    "index_name_ko": "WISE 대형고배당10 TR 지수",
    "index_name_en": "WISE Large Cap High Dividend 10 TR Index",
    "provider": "FnGuide(Wise)",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 10영업일 전",
    "effective_date_rule": "5월 선물옵션 만기일 익주 두번째 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "30영업일 일평균 거래대금 1억 미만, 30영업일 일평균 시가총액 1000억 미만, 외국인 지분제한 종목",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "기업합병, 기업분할, 관리종목 지정, 화의신청 등 이벤트 발생 시 특별변경",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "유가증권시장 보통주 중 [유동시가총액 × 현금배당총액] SCORE 상위 10종목"
  },
  {
    "file": "45_FG_SEMI_TOP10_capped.pdf",
    "index_name_ko": "FnGuide 반도체 TOP10 Capped 지수",
    "index_name_en": "FnGuide Semiconductor TOP10 Capped Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "고정비중",
    "frequency": "연2회",
    "review_period_months": 1,
    "review_date_rule": "매년 3월, 9월 말 마지막 영업일 (FnGuide 반도체 TOP10 지수와 동일)",
    "effective_date_rule": "매년 4월, 10월 선물옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC, 유동비율10%미만",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "해당없음(Constant Mix 방식으로 일별 비중 재조정)",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "FnGuide 반도체 TOP10 지수와 동일 구성종목, Constant Mix 방식으로 상위2종목 각25%, 하위8종목 50% 매일 재조정"
  },
  {
    "file": "46_FG_HY_DIV.pdf",
    "index_name_ko": "FnGuide 코리아 고배당 지수",
    "index_name_en": "FnGuide KOREA High Dividend Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매년 4월, 10월 마지막 영업일",
    "effective_date_rule": "매년 5월, 11월 선물옵션 만기일(D) 이후 2영업일째(D+2)",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 상장폐지확정종목, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC, 유동주식비율10%미만, 시가총액5000억미만, 60영업일평균거래대금10억미만",
    "ceiling_pct": 7,
    "ceiling_effective_days": null,
    "adhoc_trigger": "당해연도 주주환원정책 변화, 재무 데이터 미반영 회계손실, 영업손실 확대 시 지수위원회 검토",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "재무건전성+배당성장성 충족 종목 중 예상배당수익률 상위30종목"
  },
  {
    "file": "46_ISELECT_KDEF_TOP10.pdf",
    "index_name_ko": "iSelect K방산TOP10 지수",
    "index_name_en": "iSelect K Defense TOP10 Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일(심사일) 기준 20영업일 평균",
    "effective_date_rule": "매년 6월, 12월 옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 우선주, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC, 1년간 상장폐지실질심사대상",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "지수 구성 종목 비중 30% 초과 또는 운용상 중대한 사유 시 지수위원회 검토 후 Ceiling 비중 재조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "NLP 키워드 스코어 상위 80개 평균 초과 종목을 방위산업 유니버스로 선정 후 스코어 상위 10종목 최종 편입"
  },
  {
    "file": "47_KRX_AUTO.pdf",
    "index_name_ko": "KRX 자동차",
    "index_name_en": "KRX Automobile",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일",
    "effective_date_rule": "KOSPI200 선물 9월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "부동산투자회사, 사회기반시설투융자회사",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "부적합종목 제외, 신규상장, 합병, 기업분할",
    "june_2026_rebalancing": "N",
    "sector_count": 1,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": 95,
    "liq_pct": 90,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "48_KRX_HealthCare.pdf",
    "index_name_ko": "KRX 헬스케어",
    "index_name_en": "KRX HealthCare",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일",
    "effective_date_rule": "KOSPI200 선물 9월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "부동산투자회사, 사회기반시설투융자회사",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "부적합종목 제외, 신규상장, 합병, 기업분할",
    "june_2026_rebalancing": "N",
    "sector_count": 1,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": 95,
    "liq_pct": 90,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "49_KRX_BANK.pdf",
    "index_name_ko": "KRX 은행",
    "index_name_en": "KRX Bank",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일",
    "effective_date_rule": "KOSPI200 선물 9월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "부동산투자회사, 사회기반시설투융자회사",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "부적합종목 제외, 신규상장, 합병, 기업분할",
    "june_2026_rebalancing": "N",
    "sector_count": 1,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "4_KRX_KO200_WC.pdf",
    "index_name_ko": "코스피 200 타겟 15% 위클리 커버드콜 지수",
    "index_name_en": "KOSPI 200 Target 15% Weekly Covered Call Index",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "옵션전략",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "해당없음",
    "effective_date_rule": "해당없음",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "50_ISELECT_BIO_HealthCare.pdf",
    "index_name_ko": "iSelect 바이오헬스케어 PR 지수",
    "index_name_en": "iSelect Bio Healthcare PR Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일",
    "effective_date_rule": "매년 6월, 12월 코스피 옵션 만기일 이후",
    "rebalancing_window_days": null,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 우선주, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC",
    "ceiling_pct": 8,
    "ceiling_effective_days": null,
    "adhoc_trigger": "상장폐지, 합병/분할 등 기업이벤트로 종목수 감소 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "키워드스코어 상위 30위 이내"
  },
  {
    "file": "51_KRX_ESG_S.pdf",
    "index_name_ko": "KRX ESG 사회책임경영(S) 지수",
    "index_name_en": "KRX ESG Social Responsibility (S) Index",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 120,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": 36,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일",
    "effective_date_rule": "KOSPI200 선물 12월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "부동산투자회사, 사회기반시설투융자회사, KCGS ESG 평가대상 외 종목",
    "ceiling_pct": 27,
    "ceiling_effective_days": null,
    "adhoc_trigger": "부적합종목 발생 시 KRX TMI 기준 준용",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": 90,
    "keep_buffer": 25,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "52_KRX_REITs.pdf",
    "index_name_ko": "KRX 부동산리츠인프라 지수",
    "index_name_en": "KRX Real Estate REITs Infrastructure Index",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "정기변경일이 속한 월의 전월 최종 매매거래일",
    "effective_date_rule": "KOSPI200 선물 6월, 12월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 3,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목, 정리매매종목, 투자주의환기종목, 증권투자회사, 선박투자회사, DR, 집합투자기구, 외국주권, SPAC",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "신규상장 리츠인프라 종목이 상장일로부터 15거래일 경과 후 기준 충족 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "53_FG_NewEnergy.pdf",
    "index_name_ko": "FnGuide K-신재생에너지 플러스 지수",
    "index_name_en": "FnGuide K-Renewable Energy Plus Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": 1,
    "review_date_rule": "매년 5, 11월 마지막 영업일",
    "effective_date_rule": "6, 12월 선물옵션 만기일(D) 이후 2영업일째(D+2)",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC, 최근2사업연도연속적자, 최근2사업연도연속자본잠식",
    "ceiling_pct": 8,
    "ceiling_effective_days": null,
    "adhoc_trigger": "종목수 20종목 미만 감소 시 이벤트 적용일(D) 이후 2영업일(D+2)",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "키워드유사도 0.1 이상, 시총 상위 30위"
  },
  {
    "file": "54_ISELECT_SEMI_Equip.pdf",
    "index_name_ko": "iSelect AI반도체핵심장비 지수",
    "index_name_en": "iSelect AI Semiconductor Core Equipment Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일",
    "effective_date_rule": "매년 6월, 12월 옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 우선주, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC, 최근1년간상장폐지실질심사대상",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "상장폐지, 합병 등 기업이벤트로 편입종목이 10종목 미만 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "상위 50종목 스코어 평균값 초과"
  },
  {
    "file": "55_ISELECT_AI_ROBOT.pdf",
    "index_name_ko": "iSelect AI&로봇 지수",
    "index_name_en": "iSelect AI & Robot Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일",
    "effective_date_rule": "매년 6월, 12월 옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC, 우선주, 최근1년간상장폐지실질심사대상",
    "ceiling_pct": 7,
    "ceiling_effective_days": null,
    "adhoc_trigger": "상장폐지, 합병/분할 등 기업이벤트",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "AI산업·로봇산업 상위 80위 스코어 평균값 초과"
  },
  {
    "file": "56_FG_AUTO_TOP3.pdf",
    "index_name_ko": "FnGuide 자동차 TOP3 플러스 지수",
    "index_name_en": "FnGuide Automobile TOP3 Plus Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 13,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매년 5, 11월 마지막 영업일",
    "effective_date_rule": "6, 12월 선물옵션 만기일(D) 이후 2영업일째(D+2)",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC",
    "ceiling_pct": 25,
    "ceiling_effective_days": 3,
    "adhoc_trigger": "개별 종목 비중이 월말 종가 기준 30% 초과 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "FICS 자동차&부품 시총 상위 3종목 + 자동차소부장지수 내 시총 상위 10종목"
  },
  {
    "file": "57_DJ_DIV_30.pdf",
    "index_name_ko": "다우존스 한국 배당 30 지수",
    "index_name_en": "Dow Jones Korea Dividend 30 Index",
    "provider": "S&P Dow Jones Indices",
    "provider_type": "기타",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매년 4월말·10월말 기준, 5·11월 두번째 목요일",
    "effective_date_rule": "6월·12월 두번째 목요일 종료 후 적용",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": 4,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "58_KRX_BIO_TOP10.pdf",
    "index_name_ko": "KRX 바이오 TOP 10 지수",
    "index_name_en": "KRX Bio TOP 10 Index",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "심사연도 1월, 7월 최종 매매거래일",
    "effective_date_rule": "KOSPI200 선물 3월, 9월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 정리매매종목, 투자주의환기종목, 집합투자기구, 외국주권, 부동산투자회사, 선박투자회사, 사회기반시설투융자회사, SPAC",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "부적합종목 제외 시 예비종목 편입",
    "june_2026_rebalancing": "N",
    "sector_count": 1,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": 80,
    "keep_buffer": 40,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "59_KRX_IT.pdf",
    "index_name_ko": "KRX 정보기술",
    "index_name_en": "KRX Information Technology",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일",
    "effective_date_rule": "KOSPI200 선물 9월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "부동산투자회사, 사회기반시설투융자회사",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "부적합종목 제외, 신규상장, 합병, 기업분할",
    "june_2026_rebalancing": "N",
    "sector_count": 1,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": 95,
    "liq_pct": 90,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "5_ISELECT_AI_power.pdf",
    "index_name_ko": "iSelect AI전력핵심설비 지수",
    "index_name_en": "iSelect AI Power Core Equipment Index",
    "provider": "NH투자증권(iSelect)",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일",
    "effective_date_rule": "매년 6월, 12월 옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 우선주, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC, 최근1년간상장폐지실질심사대상",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "편입종목이 10종목 미만 시 차순위 종목 대체, 비중 30% 초과 시 지수위원회 검토",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "스코어 ≥ 최상위 스코어 × 0.1"
  },
  {
    "file": "60_FG_SECU.pdf",
    "index_name_ko": "FnGuide 보안 지수",
    "index_name_en": "FnGuide Security Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "매년 5, 11월 마지막 영업일",
    "effective_date_rule": "6, 12월 KOSPI200 선물 최종거래일이 속하는 주의 다음 주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정종목, 증권투자회사, 부동산투자회사, 선박투자회사",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "MKF지수 방법 준용",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "누적시총비중 95% 이내"
  },
  {
    "file": "61_FG_JIJOO.pdf",
    "index_name_ko": "FnGuide 지주회사 지수",
    "index_name_en": "FnGuide Holdings Company Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 30,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "매년 1월, 7월 마지막 영업일",
    "effective_date_rule": "2월, 8월 선물옵션만기일 다음주 첫 영업일(D)",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "금융지주회사, 중간지주회사, 관리종목, 상장폐지확정종목, 유동주식비율 0인 종목",
    "ceiling_pct": 8,
    "ceiling_effective_days": null,
    "adhoc_trigger": "지주사 전환 공시 발표 종목은 매 분기말 지수심의위원회 검토 후 특별편입 가능",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "유동시총 상위 30위"
  },
  {
    "file": "62_FG_HY_Plus.pdf",
    "index_name_ko": "FnGuide 고배당 Plus 지수",
    "index_name_en": "FnGuide High Dividend Plus Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 50,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": 2,
    "review_date_rule": "매년 5, 11월말 마지막 영업일",
    "effective_date_rule": "6, 12월 선물옵션 만기일 익주 첫 번째 영업일(D)부터 3영업일간",
    "rebalancing_window_days": 3,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "선박투자회사, ETF, 관리종목, 상장폐지확정종목",
    "ceiling_pct": 27,
    "ceiling_effective_days": 4,
    "adhoc_trigger": "개별 종목 비중이 종가 기준 30% 초과 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "예상배당수익률 상위 30% 이내, 최대 50종목"
  },
  {
    "file": "63_KRX_KO200_Heavy.pdf",
    "index_name_ko": "KOSPI 200 중공업 섹터지수",
    "index_name_en": "KOSPI 200 Heavy Industry Sector Index",
    "provider": "한국거래소(KRX)",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "KOSPI200 6월/12월 정기심사에서 선정된 KOSPI200 구성종목",
    "effective_date_rule": "KOSPI200 선물 6월/12월 결제월 최종거래일의 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "KOSPI200 수시변경에 연동",
    "june_2026_rebalancing": "Y",
    "sector_count": 1,
    "sector_map_type": "없음",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "64_DeepSearch_NUC_TOP10.pdf",
    "index_name_ko": "DeepSearch 원자력 Top10 지수",
    "index_name_en": "DeepSearch Nuclear-Power Top 10 Index",
    "provider": "DeepSearch",
    "provider_type": "기타",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연2회",
    "review_period_months": 2,
    "review_date_rule": "매년 5, 11월 마지막 영업일",
    "effective_date_rule": "익월 선물옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 상장폐지확정종목, 정리매매종목, 선박투자회사, 인프라투자회사, REITs, ETF, ETN, SPAC, 지주회사, 연속적자기업, 지배구조리스크기업",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "개별 종목 비중이 5거래일 연속 30% 초과 시; 원자력 관련 중요 종목 중대 이벤트 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "AI Score 상위 20종목 중 AI Score(50%)+시총(50%) 상위 10종목"
  },
  {
    "file": "65_KEDI_Hyundai_physicalAI.pdf",
    "index_name_ko": "KEDI 현대차고정피지컬AI 지수",
    "index_name_en": "KEDI Hyundai Motor Fixed Physical AI Index",
    "provider": "KEDI(한국경제신문지수)",
    "provider_type": "KEDI",
    "index_type": "주식",
    "target_n": 15,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "시총가중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "비중결정일 5영업일 전(D-5) 종목선정, 1·4·7·10월 마지막 영업일 비중결정",
    "effective_date_rule": "비중결정일(D) 기준 D+3 영업일에 정기변경 수행",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "우선주, 유사도 0.5 이하 종목 제외",
    "ceiling_pct": 25,
    "ceiling_effective_days": 3,
    "adhoc_trigger": "개별 종목 비중이 T-9~T 10영업일 연속 30% 초과 시; 현대차 비중 15% 하회 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "LLM 유사도 점수 상위 49종목 중 시총 상위 14종목 + 현대차 고정"
  },
  {
    "file": "66_Bloomberg_SamsungTOP3_Bond.pdf",
    "index_name_ko": "블룸버그 블렌디드 삼성그룹 고정 TOP3 주식 및 한국채권 지수",
    "index_name_en": "Bloomberg Blended Samsung Group Fixed Top 3 Equity and Korean Bond Index",
    "provider": "Bloomberg Index Services Limited (BISL)",
    "provider_type": "기타",
    "index_type": "채권혼합",
    "target_n": 4,
    "market_scope": "KOSPI",
    "weight_method": "고정비중",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "매 영업일 재산정",
    "effective_date_rule": "매 영업일 재산정",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "삼성전자 20%, 삼성바이오로직스 10%, 삼성SDI 10%, KIS 3Y KTB Futures Hedged Index 60% 고정"
  },
  {
    "file": "67_ISELECT_NUC_SMR.pdf",
    "index_name_ko": "iSelect K원자력SMR 지수",
    "index_name_en": "iSelect K Nuclear SMR Index",
    "provider": "NH투자증권 iSelect",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 15,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "분기4회",
    "review_period_months": 1,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일(심사일) 기준 20영업일 평균 시총·거래대금",
    "effective_date_rule": "매년 3·6·9·12월 옵션 만기일 익주 첫 영업일에 정기변경",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 우선주, 선박투자회사, REITs, 인프라투자회사, ETF&ETN, SPAC, 최근 1년 상장폐지실질심사대상",
    "ceiling_pct": 10,
    "ceiling_effective_days": null,
    "adhoc_trigger": "구성종목 비중 30% 초과 또는 운용상 중대한 사유",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "NLP 스코어 상위 20종목 중 투자적정성 충족 상위 15종목"
  },
  {
    "file": "68_WISE_SamsungGroup_value.pdf",
    "index_name_ko": "WISE삼성그룹밸류인덱스",
    "index_name_en": "WISE Samsung Group Value Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "분기4회",
    "review_period_months": 3,
    "review_date_rule": "매년 2·5·8·11월 마지막 영업일 기준",
    "effective_date_rule": "매년 3·6·9·12월 주가선물·옵션 만기일에 정기변경",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": 3,
    "excluded_types": "선박·부동산투자회사, ETF, REITs, 관리종목, 투자유의종목, 정리매매종목",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "합병·분할·상장폐지·관리종목 지정 등 기업이벤트",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "공정거래위원회 삼성그룹 계열회사 전체 (상장 보통주)"
  },
  {
    "file": "69_ISELECT_Space_Aero.pdf",
    "index_name_ko": "iSelect 우주항공UAM 지수",
    "index_name_en": "iSelect Space Aerospace UAM Index",
    "provider": "NH투자증권 iSelect",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 18,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일(심사일) 기준 직전 3개월 평균 시총·거래대금",
    "effective_date_rule": "매년 5월·11월 첫 영업일에 정기변경",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 최근 1년 상장폐지실질심사대상",
    "ceiling_pct": 10,
    "ceiling_effective_days": null,
    "adhoc_trigger": "특정 종목 비중 10% 초과 시",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "AI 키워드 필터 스코어 상위 100위 평균 이상 종목"
  },
  {
    "file": "6_FG_K_SEMI.pdf",
    "index_name_ko": "FnGuide K-반도체 지수",
    "index_name_en": "FnGuide K-Semiconductor Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 20,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "매년 5·11월 말 마지막 영업일 기준 종목 선정",
    "effective_date_rule": "매년 6·12월 선물옵션 만기일(D) 이후 D+4·D+5 양일에 걸쳐 변경",
    "rebalancing_window_days": 2,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC, 당기순이익 2년 연속 적자",
    "ceiling_pct": 25,
    "ceiling_effective_days": 2,
    "adhoc_trigger": "전월 마지막 영업일 또는 해당월 만기일 종가 기준 30% 초과 종목 존재 시 D+2에 25%로 비중제한",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": 20,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "FICS 반도체 해당 또는 반도체 매출비중 10% 초과 종목 중 유동시총 상위 20종목"
  },
  {
    "file": "70_ISELECT_B_SEMI.pdf",
    "index_name_ko": "iSelect 비메모리반도체 지수",
    "index_name_en": "iSelect Non-Memory Semiconductor Index",
    "provider": "NH투자증권 iSelect",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": 34,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "정기변경일 전월 마지막 영업일(심사일) 직전 3개월 평균 시총·거래대금",
    "effective_date_rule": "매년 6·12월 코스피200 선물 만기일(T)로부터 T+5 영업일에 정기변경",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 최근 1년 상장폐지실질심사대상, 유동비율 하위 10%",
    "ceiling_pct": 15,
    "ceiling_effective_days": null,
    "adhoc_trigger": "특정 종목 비중 15% 초과 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "AI 키워드 필터 스코어 기반 정성평가 통과 종목"
  },
  {
    "file": "71_FG_SEMI_VALUE.pdf",
    "index_name_ko": "FnGuide 반도체 밸류체인 지수",
    "index_name_en": "FnGuide Semiconductor Value Chain Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 70,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매년 6·12월 정기개편 달 마지막 영업일 기준 종목 선정",
    "effective_date_rule": "매년 6·12월 선물옵션 만기일(D) 이후 D+2 영업일에 정기변경",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의·투자경고·투자위험 지정, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "MKF500 내 FICS IT·반도체·하드웨어 섹터 종목 중 2차전지 매출 제외, 유동시총 상위 70종목"
  },
  {
    "file": "73_KRX_HY_WeeklyCC.pdf",
    "index_name_ko": "코스피 고배당 위클리 콜매도 ATM 지수",
    "index_name_en": "KOSPI High Dividend Weekly Call Sold ATM Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "옵션전략",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "해당없음(옵션전략지수)",
    "effective_date_rule": "매 결제주·결제월 최종거래일 코스피200 종가 산출 시점에 산출대상 콜옵션 선정",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "74_KRX_300.pdf",
    "index_name_ko": "KRX 300 지수",
    "index_name_en": "KRX 300 Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 300,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 6,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일(심사기준일), 심사대상기간은 심사기준일 포함 소급 최근 6개월",
    "effective_date_rule": "KOSPI 200 선물 6·12월 결제월 최종거래일 다음 매매거래일(정기변경일)",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": 6,
    "excluded_types": "부동산투자회사, 사회기반시설투융자회사, 자본잠식 종목",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "TMI 부적합종목 제외, 신규상장특례(시총 상위 50위 이내), 합병·기업분할",
    "june_2026_rebalancing": "Y",
    "sector_count": 10,
    "sector_map_type": "KRX통합",
    "sector_min_pct": "N",
    "cap_threshold": 80,
    "liq_pct": 80,
    "keep_buffer": 110,
    "new_buffer": null,
    "tertiary_mode": "global",
    "small_excl_rank": null,
    "large_cap_special_rank": 100,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "75_ISELECT_SecondaryBattery.pdf",
    "index_name_ko": "iSelect 2차전지 지수",
    "index_name_en": "iSelect Secondary Battery Index",
    "provider": "NH투자증권 iSelect",
    "provider_type": "iSelect",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "분기4회",
    "review_period_months": 3,
    "review_date_rule": "정기변경일 직전 달 마지막 영업일(심사일) 직전 3개월 평균 시총(1,000억 이상)·거래대금(3억 이상)",
    "effective_date_rule": "매년 3·6·9·12월 코스피200 선물만기일 2영업일 이후 정기변경",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리대상종목, 정리매매종목, 유동주식비율 10% 미만, 최근 1년 상장폐지실질심사대상, 유동비율 하위 10%",
    "ceiling_pct": 9,
    "ceiling_effective_days": null,
    "adhoc_trigger": "특정 종목 비중 9% 초과 시",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "AI 키워드 필터 스코어 상위 100위 평균 이상 종목(시총 1,000억 미만 제외)"
  },
  {
    "file": "76_KRX_SEMI_CC.pdf",
    "index_name_ko": "AI 반도체 위클리 고정 30% 커버드콜 지수",
    "index_name_en": "AI Semiconductor Weekly Fixed 30% Covered Call Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "옵션전략",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "해당없음(옵션전략지수)",
    "effective_date_rule": "매 결제주·결제월 최종거래일 코스피200 종가 산출 시점에 산출대상 콜옵션 선정",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "77_KRX_KOSPI.pdf",
    "index_name_ko": "코스피지수",
    "index_name_en": "KOSPI Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "시총가중",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "유가증권시장 상장 보통주 전체(신규상장 후 1매매거래일 경과 후 편입)",
    "effective_date_rule": "신규상장 후 1매매거래일 경과일에 자동 편입",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음(전 종목 편입)",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "79_KRX_KO200_WC.pdf",
    "index_name_ko": "코스피 200 타겟 7% 위클리 커버드콜 지수",
    "index_name_en": "KOSPI 200 Target 7% Weekly Covered Call Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "옵션전략",
    "target_n": null,
    "market_scope": "KOSPI",
    "weight_method": "기타",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "해당없음(옵션전략지수)",
    "effective_date_rule": "매 결제주·결제월 최종거래일 코스피200 종가 산출 시점에 산출대상 콜옵션 선정",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "7_FG_Samsung_Group.pdf",
    "index_name_ko": "FnGuide 삼성그룹 지수",
    "index_name_en": "FnGuide Samsung Group Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 3,
    "review_date_rule": "개편 직전 월의 마지막 영업일(Review일)",
    "effective_date_rule": "6·12월 2번째 영업일부터 6번째 영업일까지 5영업일에 걸쳐 순차적으로 정기변경",
    "rebalancing_window_days": 5,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "선박·부동산투자회사, ETF, REITs, 관리종목, 투자유의종목, 정리매매종목",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "합병·분할·상장폐지·관리종목 지정 등 기업이벤트",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "공정거래위원회 삼성그룹 계열회사 중 시총 1조 이상, 거래대금 50억 이상, 유동주식비율 10% 이상 등 충족 종목"
  },
  {
    "file": "80_FG_HY_Focus.pdf",
    "index_name_ko": "FnGuide 고배당포커스 지수",
    "index_name_en": "FnGuide High Dividend Focus Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 80,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "연1회",
    "review_period_months": 4,
    "review_date_rule": "매 4월말 종가 기준 종목 선정",
    "effective_date_rule": "매 5월 선물옵션 만기일 익주 두번째~세번째 영업일(2영업일 50%씩 순차 개편)",
    "rebalancing_window_days": 2,
    "float_rate_min": null,
    "listing_period_months": 3,
    "excluded_types": "관리종목, 투자유의종목, 상장폐지확정",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "구성종목 60종목 미만 감소 시 수시 추가 편입 가능",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "MKF500 중 현금배당수익률 상위 30% 또는 MKF 중대형 유동시총 비중 10% 이상 의무편입, 최종 현금배당수익률 순위 상위 80종목"
  },
  {
    "file": "81_KRX_KO200_IT_Sector.pdf",
    "index_name_ko": "KRX 섹터지수 (KOSPI 200·KOSDAQ 150·KRX 300 섹터)",
    "index_name_en": "KRX Sector Indices (KOSPI 200 / KOSDAQ 150 / KRX 300 Sector)",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "KOSPI 200·KOSDAQ 150·KRX 300 6·12월 정기심사 결과 준용",
    "effective_date_rule": "KOSPI 200 선물 6·12월 결제월 최종거래일 다음 매매거래일",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "해당없음(모지수 구성종목 전체가 대상)",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "편입비중 과도 상승 시 정기조정 전 수시 CAP Factor 조정 가능",
    "june_2026_rebalancing": "Y",
    "sector_count": 1,
    "sector_map_type": "KRX통합",
    "sector_min_pct": "N",
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "82_KRX_value_up.pdf",
    "index_name_ko": "코리아 밸류업 지수",
    "index_name_en": "Korea Value-up Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 100,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": 12,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일(심사기준일), 시장대표성·유동성 심사대상기간 최근 12개월",
    "effective_date_rule": "KOSPI 200 선물 6월 결제월 최종거래일 다음 매매거래일(연 1회)",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": 12,
    "excluded_types": "관리종목, 투자주의환기종목, 정리매매종목, 부동산투자회사, 선박투자회사, 사회기반시설투융자회사, 기업인수목적회사, 자본잠식",
    "ceiling_pct": 15,
    "ceiling_effective_days": null,
    "adhoc_trigger": "편입비중 과도 상승 시 정기조정 전 수시 CAP Factor 조정 가능",
    "june_2026_rebalancing": "Y",
    "sector_count": 10,
    "sector_map_type": "KRX통합",
    "sector_min_pct": "N",
    "cap_threshold": 400,
    "liq_pct": 80,
    "keep_buffer": 440,
    "new_buffer": 400,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "83_FG_AI_SEMI_Core.pdf",
    "index_name_ko": "FnGuide K-AI반도체 코어테크 지수",
    "index_name_en": "FnGuide K-AI Semiconductor Core Technology Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 20,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": 12,
    "review_date_rule": "매년 5·11월 마지막 영업일 기준 종목 선정(과거 1년치 리포트·보고서 기반 TF-IDF 코사인 유사도)",
    "effective_date_rule": "매년 6·12월 선물옵션 만기일(D) 이후 D+2 영업일에 정기변경",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정, 선박투자회사, 인프라투자회사, 해외주, REITs, ETF, ETN, SPAC",
    "ceiling_pct": 20,
    "ceiling_effective_days": 3,
    "adhoc_trigger": "매월 마지막 영업일 종가 기준 3영업일 연속(T-2,T-1,T) 30% 초과 시 T+3에 25%로 재조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "FICS IT 섹터 내 AI반도체 키워드 합산스코어(공시 0.5+리포트 0.5) 상위 20종목"
  },
  {
    "file": "84_NICE_SEMI_TOP2.pdf",
    "index_name_ko": "NICE K 반도체 TOP2 MAX+ 지수",
    "index_name_en": "NICE K Semiconductor TOP2 MAX+ Index",
    "provider": "NICE 피앤아이",
    "provider_type": "기타",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "매 3·6·9·12월 마지막 영업일 기준 2영업일 전 종가 기준 종목 및 비중 결정",
    "effective_date_rule": "종목선정일(마지막 영업일 기준 2영업일 전) 기준 3영업일 후 적용",
    "rebalancing_window_days": 1,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목, 투자주의환기종목, 상장폐지확정, 우선주, 선박투자회사, REITs, 인프라투자회사, ETF, ETN, SPAC",
    "ceiling_pct": 27.5,
    "ceiling_effective_days": 3,
    "adhoc_trigger": "개별 종목 편입 비중이 5영업일 연속(T-4~T) 30% 초과 시 T+3에 27.5%로 재조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "NICS 반도체·디스플레이·하드웨어 중분류 중 반도체 관련 매출 존재 종목의 시총 상위 10종목"
  },
  {
    "file": "86_FG_Growth.pdf",
    "index_name_ko": "FnGuide 성장 지수",
    "index_name_en": "FnGuide Growth Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매 5,11월말 마지막 영업일",
    "effective_date_rule": "6,12월 선물옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": null,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "선박·부동산투자회사,ETF,REITs,SPAC,관리종목,투자유의종목,상장폐지확정,3사업연도연속적자+매출200억미만",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "MKF500 기반 GIF(Growth Inclusion Factor) 산출"
  },
  {
    "file": "87_KO200_ESG.pdf",
    "index_name_ko": "코스피 200 ESG 지수",
    "index_name_en": "KOSPI 200 ESG Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 100,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": 36,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일",
    "effective_date_rule": "KOSPI200 선물시장 12월 결제월 최종거래일 다음 매매거래일",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "KCGS ESG평가대상 아닌 종목,ESG통합점수 0점,관리종목,도박·담배·주류·군수산업 매출비중20%이상",
    "ceiling_pct": 27,
    "ceiling_effective_days": null,
    "adhoc_trigger": "KOSPI200 수시변경 제외 또는 KCGS ESG등급 수시조정(통합등급 B+미만+평균미만)",
    "june_2026_rebalancing": "N",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": 10,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "최근 3개년 ESG 통합점수 평균 상위 100종목"
  },
  {
    "file": "88_KRX_KOSPI_Large.pdf",
    "index_name_ko": "KOSPI 100 지수",
    "index_name_en": "KOSPI 100 Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 100,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "KOSPI200 6월/12월 정기심사에서 산정한 일평균시가총액 기준",
    "effective_date_rule": "KOSPI200 선물시장 6월/12월 결제월 최종거래일 다음 매매거래일",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "KOSPI200 미편입 종목",
    "ceiling_pct": 30,
    "ceiling_effective_days": null,
    "adhoc_trigger": "KOSPI200 수시변경 제외 종목이 구성종목인 경우 예비종목 1순위 편입",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": 120,
    "new_buffer": 80,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "KOSPI200 구성종목 중 일평균시가총액 상위 100종목"
  },
  {
    "file": "89_KRX_KO50.pdf",
    "index_name_ko": "KOSPI 50 지수",
    "index_name_en": "KOSPI 50 Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 50,
    "market_scope": "KOSPI",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "KOSPI200 6월/12월 정기심사에서 산정한 일평균시가총액 기준",
    "effective_date_rule": "KOSPI200 선물시장 6월/12월 결제월 최종거래일 다음 매매거래일",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": "KOSPI200 미편입 종목",
    "ceiling_pct": 30,
    "ceiling_effective_days": null,
    "adhoc_trigger": "KOSPI200 수시변경 제외 종목이 구성종목인 경우 예비종목 1순위 편입",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": 120,
    "new_buffer": 80,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "KOSPI200 구성종목 중 일평균시가총액 상위 50종목"
  },
  {
    "file": "8_KRX_value_up.pdf",
    "index_name_ko": "코리아 밸류업 지수",
    "index_name_en": "Korea Value-up Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "주식",
    "target_n": 100,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연1회",
    "review_period_months": 12,
    "review_date_rule": "정기변경일이 속한 월의 전전월 최종 매매거래일(심사기준일)",
    "effective_date_rule": "KOSPI200 선물시장 6월 결제월 최종거래일 다음 매매거래일",
    "rebalancing_window_days": null,
    "float_rate_min": 10,
    "listing_period_months": 12,
    "excluded_types": "관리종목,투자주의환기종목,정리매매종목,부동산·선박·사회기반시설투융자회사,기업인수목적회사,자본잠식",
    "ceiling_pct": 15,
    "ceiling_effective_days": null,
    "adhoc_trigger": "관리종목지정,투자주의환기종목지정,상장폐지결정,기업분할 등 부적합종목 발생",
    "june_2026_rebalancing": "Y",
    "sector_count": 10,
    "sector_map_type": "KRX통합",
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": 80,
    "keep_buffer": 110,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "심사대상종목 중 일평균시가총액 상위 400위(기존구성종목 잔류버퍼 440위)"
  },
  {
    "file": "90_KRX_Valueup_WC.pdf",
    "index_name_ko": "코리아 밸류업 위클리 커버드콜 30% 지수",
    "index_name_en": "Korea Value-up Weekly Covered Call 30% Index",
    "provider": "KRX",
    "provider_type": "KRX",
    "index_type": "옵션전략",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "코스피200 옵션 결제주/결제월 최종거래일 기준 자동선정",
    "effective_date_rule": "옵션 결제주/결제월별 자동 롤오버",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "91_FG_SE_SKH_BOND.pdf",
    "index_name_ko": "FnGuide 삼성전자 & SK하이닉스 채권혼합 지수",
    "index_name_en": "FnGuide Samsung Electronics & SK Hynix Bond Balanced Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "채권혼합",
    "target_n": 2,
    "market_scope": "KOSPI",
    "weight_method": "고정비중",
    "frequency": "없음(파생)",
    "review_period_months": null,
    "review_date_rule": "해당없음(Constant Mix 일일조정)",
    "effective_date_rule": "매일 삼성전자25%:SK하이닉스25%:국고통안채50% 비중 유지",
    "rebalancing_window_days": 1,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "영업손실 확대 등 투자자 보호 필요 시 지수위원회 검토",
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "기타",
    "primary_threshold": "삼성전자25%+SK하이닉스25%+FnGuide국고통안채지수50% 고정비중 혼합"
  },
  {
    "file": "92_FG_Value.pdf",
    "index_name_ko": "FnGuide 기업가치 지수",
    "index_name_en": "FnGuide Enterprise Value Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 40,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "동일가중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "매년 2,5,8,11월 마지막 영업일",
    "effective_date_rule": "3,6,9,12월 선물옵션 만기일(D) 이후 2영업일째(D+2)",
    "rebalancing_window_days": 2,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목,투자유의환기종목,상장폐지확정,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC,시총1조원미만,20영업일평균거래대금20억미만",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "주주환원정책 변화,미반영 회계손실,운용상 중대편출사유,영업손실 확대 등 지수위원회 검토",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "재무요건(PBR1.3이하,PER25이하,주주환원3%이상,2개년연속흑자) 충족 후 시총상위40종목"
  },
  {
    "file": "93_FG_IT_Plus.pdf",
    "index_name_ko": "FnGuide IT플러스 지수",
    "index_name_en": "FnGuide IT PLUS Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "유동시총가중",
    "frequency": "연2회",
    "review_period_months": null,
    "review_date_rule": "매년 5,11월 말 마지막 영업일",
    "effective_date_rule": "6,12월 선물옵션 만기일 익주 첫 영업일",
    "rebalancing_window_days": null,
    "float_rate_min": 10,
    "listing_period_months": 3,
    "excluded_types": "관리종목,투자유의환기종목,상장폐지확정,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC,시총3000억미만,60영업일평균거래대금10억미만",
    "ceiling_pct": 25,
    "ceiling_effective_days": null,
    "adhoc_trigger": "미반영 회계손실,운용상 중대편출사유,영업손실 확대 등 지수위원회 검토",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "MKF500 중 FICS 중분류 소프트웨어·하드웨어·반도체·디스플레이 해당 종목"
  },
  {
    "file": "94_FG_SEMI_Infra.pdf",
    "index_name_ko": "FnGuide AI반도체&인프라 지수",
    "index_name_en": "FnGuide AI Semiconductor & Infrastructure Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 24,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "고정비중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "매년 2,5,9,11월 마지막 영업일",
    "effective_date_rule": "3,6,9,12월 선물옵션 만기일(D) 이후 2영업일째(D+2)",
    "rebalancing_window_days": 2,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목,투자유의환기종목,상장폐지확정,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC,지주회사,시총4000억미만,60영업일평균거래대금10억미만",
    "ceiling_pct": 20,
    "ceiling_effective_days": null,
    "adhoc_trigger": "테마관련성 낮은 종목,미반영 회계손실,운용상 중대편출사유,영업손실 확대",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "테마풀사전정의",
    "primary_threshold": "AI반도체인프라 키워드 공시스크리닝 통과 종목 중 테마별시총1위4종목(GPU20%,HBM·인공지능·전력사업각10%)+재무스코어상위20종목"
  },
  {
    "file": "95_KRXSNP_Carbon.pdf",
    "index_name_ko": "KRX S&P 탄소효율 그린뉴딜 지수",
    "index_name_en": "KRX S&P Carbon Efficient Green New Deal Index",
    "provider": "S&P Dow Jones Indices / KRX",
    "provider_type": "기타",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "기타",
    "frequency": null,
    "review_period_months": null,
    "review_date_rule": null,
    "effective_date_rule": null,
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "9_FG_AISEMI_TOP2.pdf",
    "index_name_ko": "FnGuide AI반도체TOP2플러스 지수",
    "index_name_en": "FnGuide AI Semiconductor TOP2 Plus Index",
    "provider": "FnGuide",
    "provider_type": "FnGuide계열",
    "index_type": "주식",
    "target_n": 10,
    "market_scope": "KOSPI+KOSDAQ",
    "weight_method": "고정비중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "매년 3,6,9,12월 마지막 영업일",
    "effective_date_rule": "1,4,7,10월 옵션 만기일(D) 이후 2영업일째(D+2)",
    "rebalancing_window_days": 2,
    "float_rate_min": 10,
    "listing_period_months": null,
    "excluded_types": "관리종목,투자유의환기종목,상장폐지확정,선박투자회사,인프라투자회사,해외주,REITs,ETF,ETN,SPAC,시총2000억미만,60영업일평균거래대금20억미만",
    "ceiling_pct": 25,
    "ceiling_effective_days": 3,
    "adhoc_trigger": "개별종목 비중 3영업일 연속 30% 초과 시 T+3에 25% 수시조정",
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": "시총순위",
    "primary_threshold": "반도체섹터시총상위2종목(TOP2각25%고정)+모멘텀합산스코어상위4종목+시총상위4종목"
  },
  {
    "file": "MSCI_0_INFORMATION_DOCUMENT.pdf",
    "index_name_ko": "MSCI 방법론 세트 안내 문서",
    "index_name_en": "MSCI Information Document - Description of Methodologies Set",
    "provider": "MSCI",
    "provider_type": "MSCI",
    "index_type": "기타",
    "target_n": null,
    "market_scope": "해외",
    "weight_method": "기타",
    "frequency": null,
    "review_period_months": null,
    "review_date_rule": "개별 지수 방법론 참조",
    "effective_date_rule": "개별 지수 방법론 참조",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "MSCI_0_MSCI_Corporate_Events_Methodology_20260210.pdf",
    "index_name_ko": "MSCI 기업이벤트 방법론",
    "index_name_en": "MSCI Corporate Events Methodology (February 2026)",
    "provider": "MSCI",
    "provider_type": "MSCI",
    "index_type": "기타",
    "target_n": null,
    "market_scope": "해외",
    "weight_method": "기타",
    "frequency": null,
    "review_period_months": null,
    "review_date_rule": "해당없음(공통 방법론 문서)",
    "effective_date_rule": "해당없음(공통 방법론 문서)",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "MSCI_0_MSCI_Fundamental_Data_Methodology_20240625.pdf",
    "index_name_ko": "MSCI 펀더멘털 데이터 방법론",
    "index_name_en": "MSCI Fundamental Data Methodology (June 2024)",
    "provider": "MSCI",
    "provider_type": "MSCI",
    "index_type": "기타",
    "target_n": null,
    "market_scope": "해외",
    "weight_method": "기타",
    "frequency": null,
    "review_period_months": null,
    "review_date_rule": "해당없음(공통 방법론 문서)",
    "effective_date_rule": "해당없음(공통 방법론 문서)",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "MSCI_0_MSCI_Index_Calculation_Methodology_20260512.pdf",
    "index_name_ko": "MSCI 지수 계산 방법론",
    "index_name_en": "MSCI Index Calculation Methodology (May 2026)",
    "provider": "MSCI",
    "provider_type": "MSCI",
    "index_type": "기타",
    "target_n": null,
    "market_scope": "해외",
    "weight_method": "기타",
    "frequency": null,
    "review_period_months": null,
    "review_date_rule": "해당없음(공통 방법론 문서)",
    "effective_date_rule": "해당없음(공통 방법론 문서)",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "MSCI_0_MSCI_Index_Glossary_of_Terms_20260105.pdf",
    "index_name_ko": "MSCI 지수 용어집",
    "index_name_en": "MSCI Index Glossary of Terms (January 2026)",
    "provider": "MSCI",
    "provider_type": "MSCI",
    "index_type": "기타",
    "target_n": null,
    "market_scope": "해외",
    "weight_method": "기타",
    "frequency": null,
    "review_period_months": null,
    "review_date_rule": "해당없음(공통 방법론 문서)",
    "effective_date_rule": "해당없음(공통 방법론 문서)",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "MSCI_0_MSCI_Index_Policies_20260127.pdf",
    "index_name_ko": "MSCI 지수 정책",
    "index_name_en": "MSCI Index Policies (January 2026)",
    "provider": "MSCI",
    "provider_type": "MSCI",
    "index_type": "기타",
    "target_n": null,
    "market_scope": "해외",
    "weight_method": "기타",
    "frequency": null,
    "review_period_months": null,
    "review_date_rule": "해당없음(공통 방법론 문서)",
    "effective_date_rule": "해당없음(공통 방법론 문서)",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "MSCI_1_MSCI_Global_Industry_Classification_Standard_GICS_Methodology_20250220.pdf",
    "index_name_ko": "MSCI GICS 글로벌 산업분류 방법론",
    "index_name_en": "MSCI Global Industry Classification Standard (GICS) Methodology",
    "provider": "MSCI",
    "provider_type": "MSCI",
    "index_type": "기타",
    "target_n": null,
    "market_scope": "해외",
    "weight_method": "기타",
    "frequency": "연1회",
    "review_period_months": null,
    "review_date_rule": "연간 GICS 구조 검토(GICS Structure Review)",
    "effective_date_rule": "GICS 구조 변경 공지 후 적용",
    "rebalancing_window_days": null,
    "float_rate_min": null,
    "listing_period_months": null,
    "excluded_types": null,
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": null,
    "june_2026_rebalancing": "해당없음",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null
  },
  {
    "file": "MSCI_1_MSCI_Global_Investable_Market_Indexes_Methodology_20260512.pdf",
    "index_name_ko": "MSCI 글로벌 투자가능시장 지수 방법론",
    "index_name_en": "MSCI Global Investable Market Indexes (GIMI) Methodology (May 2026)",
    "provider": "MSCI",
    "provider_type": "MSCI",
    "index_type": "주식",
    "target_n": null,
    "market_scope": "해외",
    "weight_method": "유동시총가중",
    "frequency": "분기4회",
    "review_period_months": null,
    "review_date_rule": "2,5,8,11월 분기별 지수 리뷰(QIR); 5월·11월은 Semi-Annual Review(SAR)로 전체재구성",
    "effective_date_rule": "리뷰 결과 공표 후 Effective Date 시초가 기준 적용",
    "rebalancing_window_days": null,
    "float_rate_min": 15,
    "listing_period_months": 3,
    "excluded_types": "FIF 0.15 미만 종목,ATVR 유동성기준 미충족(DM:20%/EM:15%),외국인보유가능한도(Foreign Room) 15% 미만,상장 3개월 미만 소규모 신규상장",
    "ceiling_pct": null,
    "ceiling_effective_days": null,
    "adhoc_trigger": "대형 IPO(10거래일 후 Early Inclusion),대형 M&A,조기편출(Early Deletion),기업이벤트에 따른 FIF/NOS 수시조정",
    "june_2026_rebalancing": "Y",
    "sector_count": null,
    "sector_map_type": null,
    "sector_min_pct": null,
    "cap_threshold": null,
    "liq_pct": null,
    "keep_buffer": null,
    "new_buffer": null,
    "tertiary_mode": null,
    "small_excl_rank": null,
    "large_cap_special_rank": null,
    "primary_metric": null,
    "primary_threshold": null,
    "fif_min": 15,
    "air_min": null,
    "buffer_zone": 33,
    "sar_months": "5,11",
    "qir_months": "2,8",
    "off_cycle_trigger": "대형 IPO 10거래일 후 Early Inclusion,대형 M&A Early Deletion/Inclusion,시장구조 변화에 따른 수시 FIF/NOS 조정"
  }
]

print(f'로드 완료: {len(ALL_INDEX_DATA)}개 지수')

## 3. 기본 통계 및 2026년 6월 리밸런싱 현황

In [ ]:
# June 2026 리밸런싱 분류
y_data  = [d for d in ALL_INDEX_DATA if str(d.get('june_2026_rebalancing','')).startswith('Y')]
n_data  = [d for d in ALL_INDEX_DATA if str(d.get('june_2026_rebalancing','')).startswith('N')]
na_data = [d for d in ALL_INDEX_DATA if not str(d.get('june_2026_rebalancing','')).startswith(('Y','N'))]

print(f'전체 지수: {len(ALL_INDEX_DATA)}개')
print(f'6월 리밸런싱 Y: {len(y_data)}개')
print(f'6월 리밸런싱 N: {len(n_data)}개')
print(f'해당없음/옵션전략: {len(na_data)}개')
print()

# 제공사별 분류
by_type = Counter(d.get('provider_type','기타') for d in ALL_INDEX_DATA)
print('제공사별 지수 수:')
for pt, cnt in sorted(by_type.items(), key=lambda x: -x[1]):
    y_cnt = sum(1 for d in ALL_INDEX_DATA if d.get('provider_type')==pt and str(d.get('june_2026_rebalancing','')).startswith('Y'))
    print(f'  {pt:<15}: 총 {cnt:>3}개 | 6월Y {y_cnt:>3}개')

## 4. 2026년 6월 리밸런싱 예정 지수 목록 (Y=57개)

In [ ]:
print(f'{"No":>3} {"지수사":<15} {"지수명(한글)":<32} {"주기":<12} {"변경적용일"}')
print('-' * 100)
for i, d in enumerate(y_data, 1):
    print(f'{i:>3} {d.get("provider",""):<15} {d.get("index_name_ko",""):<32} '
          f'{d.get("frequency",""):<12} {d.get("effective_date_rule","")}')

## 5. 지수사별 6월 리밸런싱 요약

In [ ]:
summary = defaultdict(lambda: {'Y': 0, 'N': 0, '-': 0})
for d in ALL_INDEX_DATA:
    pt = d.get('provider_type', '기타')
    val = str(d.get('june_2026_rebalancing', ''))
    if val.startswith('Y'):   summary[pt]['Y'] += 1
    elif val.startswith('N'): summary[pt]['N'] += 1
    else:                     summary[pt]['-'] += 1

print(f'{"제공사 유형":<18} {"Y":>5} {"N":>5} {"해당없음":>8} {"합계":>5}')
print('-' * 45)
total_y, total_n, total_na = 0, 0, 0
for pt, counts in sorted(summary.items()):
    y, n, na = counts['Y'], counts['N'], counts['-']
    total_y += y; total_n += n; total_na += na
    print(f'{pt:<18} {y:>5} {n:>5} {na:>8} {y+n+na:>5}')
print('-' * 45)
print(f'{"합계":<18} {total_y:>5} {total_n:>5} {total_na:>8} {total_y+total_n+total_na:>5}')

## 6. KRX 지수 Rule Engine 파라미터 분석

KRX 지수의 8단계 Rule Engine 파라미터를 분석합니다.

### 8단계 파이프라인
1. Universe Filter (유동비율, 상장기간, 제외종목)
2. Sector Classification (섹터 분류)
3. 1차 선정: cap_threshold + liq_pct (시총/거래대금 누적비중)
4. 2차 선정: keep_buffer + new_buffer (편입유지/신규편입 버퍼)
5. 3차 선정: target_n + tertiary_mode (목표종목수 채우기)
6. Merger Whitelist (합병 유지)
7. Matching (종목 확정)
8. Large-cap Special (시총 상위 50위 자동편입)

In [ ]:
# KRX 주식 지수만 필터링
krx_data = [d for d in ALL_INDEX_DATA if d.get('provider_type') == 'KRX' and d.get('index_type') == '주식']
print(f'KRX 주식 지수: {len(krx_data)}개')
print()

# Rule Engine 파라미터 출력
print(f'{"파일명":<38} {"cap":<5} {"liq":<5} {"keep":<5} {"new":<5} {"tert":<12} {"target_n":<9} {"ceiling"}')
print('-' * 100)
for d in sorted(krx_data, key=lambda x: x.get('file','')):
    print(f'{d.get("file",""):<38} '
          f'{str(d.get("cap_threshold","-")):<5} '
          f'{str(d.get("liq_pct","-")):<5} '
          f'{str(d.get("keep_buffer","-")):<5} '
          f'{str(d.get("new_buffer","-")):<5} '
          f'{str(d.get("tertiary_mode","-")):<12} '
          f'{str(d.get("target_n","-")):<9} '
          f'{str(d.get("ceiling_pct","-"))}')

## 7. Excel 파일 생성 (v2)

5개 시트로 구성된 Excel 파일을 생성합니다:
- **Sheet 1** `June2026_리밸런싱_요약`: 100개 전체 (Y/N 색상 구분)
- **Sheet 2** `June2026_Y_리밸런싱`: Y 지수만 (57개)
- **Sheet 3** `KRX_Rule_Engine_파라미터`: KRX 지수 Rule Engine 파라미터
- **Sheet 4** `전체_지수_기본정보`: 전체 지수 v2 스키마 전체 필드
- **Sheet 5** `제공사별_통계`: 제공사별 통계 요약

In [ ]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─── Styles ───
header_fill    = PatternFill('solid', fgColor='1F4E79')
header_font    = Font(bold=True, color='FFFFFF', size=10)
yes_fill       = PatternFill('solid', fgColor='C6EFCE')
no_fill        = PatternFill('solid', fgColor='FFCCCC')
na_fill        = PatternFill('solid', fgColor='FFFFCC')
alt_fill       = PatternFill('solid', fgColor='EBF3FB')
border_side    = Side(style='thin', color='BFBFBF')
thin_border    = Border(left=border_side, right=border_side, top=border_side, bottom=border_side)

def hdr(ws, row, col, val, fill=None, font=None):
    c = ws.cell(row=row, column=col, value=val)
    c.fill = fill or header_fill
    c.font = font or header_font
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border = thin_border
    return c

def cel(ws, row, col, val, fill=None, center=False):
    c = ws.cell(row=row, column=col, value=val)
    if fill: c.fill = fill
    c.alignment = Alignment(horizontal='center' if center else 'left', vertical='center', wrap_text=True)
    c.border = thin_border
    return c

def june_fill(val):
    v = str(val)
    if v.startswith('Y'): return yes_fill
    if v.startswith('N'): return no_fill
    return na_fill

wb = openpyxl.Workbook()

# ── Sheet 1: 요약 ──
ws = wb.active
ws.title = 'June2026_리밸런싱_요약'
ws.merge_cells('A1:J1')
t = ws.cell(1, 1, '2026년 6월 정기/수시 리밸런싱 예정 지수 현황 (v2)')
t.font = Font(bold=True, size=13, color='FFFFFF')
t.fill = PatternFill('solid', fgColor='1F4E79')
t.alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[1].height = 30

hdrs = ['No','파일명','지수명(한글)','지수사','유형','June2026','주기','심사기준일','변경적용일','CAP(%)']
for c, h in enumerate(hdrs, 1): hdr(ws, 2, c, h)
ws.row_dimensions[2].height = 35
for i, w in enumerate([5,28,32,18,10,8,18,35,40,10], 1):
    ws.column_dimensions[get_column_letter(i)].width = w

for idx, d in enumerate(ALL_INDEX_DATA, 1):
    row = idx + 2
    jv = str(d.get('june_2026_rebalancing',''))
    rf = june_fill(jv)
    cel(ws, row, 1, idx, rf)
    cel(ws, row, 2, d.get('file',''), rf)
    cel(ws, row, 3, d.get('index_name_ko',''), rf)
    cel(ws, row, 4, d.get('provider',''), rf)
    cel(ws, row, 5, d.get('provider_type',''), rf)
    js = 'Y' if jv.startswith('Y') else ('N' if jv.startswith('N') else '-')
    c = ws.cell(row, 6, js); c.fill = rf; c.font = Font(bold=True,size=10); c.alignment = Alignment(horizontal='center',vertical='center'); c.border = thin_border
    cel(ws, row, 7, d.get('frequency',''), rf)
    cel(ws, row, 8, d.get('review_date_rule',''), rf)
    cel(ws, row, 9, d.get('effective_date_rule',''), rf)
    cel(ws, row, 10, d.get('ceiling_pct'), rf)
    ws.row_dimensions[row].height = 40
ws.freeze_panes = 'A3'

# ── Sheet 2: Y 지수 ──
ws2 = wb.create_sheet('June2026_Y_리밸런싱')
ws2.merge_cells('A1:I1')
t2 = ws2.cell(1,1,'2026년 6월 정기변경 예정 지수 (Y=57개)')
t2.font = Font(bold=True,size=13,color='FFFFFF'); t2.fill = PatternFill('solid',fgColor='375623'); t2.alignment = Alignment(horizontal='center',vertical='center')
ws2.row_dimensions[1].height = 28
hdrs2 = ['No','파일명','지수명(한글)','지수사','유형','주기','심사기준일','변경적용일','June2026 상세']
for c,h in enumerate(hdrs2,1): hdr(ws2,2,c,h)
ws2.row_dimensions[2].height = 35
for i,w in enumerate([5,28,32,18,12,18,35,40,55],1): ws2.column_dimensions[get_column_letter(i)].width = w
y_idx = [d for d in ALL_INDEX_DATA if str(d.get('june_2026_rebalancing','')).startswith('Y')]
for idx,d in enumerate(y_idx,1):
    row = idx+2; rf = yes_fill if idx%2==1 else PatternFill('solid',fgColor='AADDAA')
    cel(ws2,row,1,idx,rf); cel(ws2,row,2,d.get('file',''),rf); cel(ws2,row,3,d.get('index_name_ko',''),rf)
    cel(ws2,row,4,d.get('provider',''),rf); cel(ws2,row,5,d.get('provider_type',''),rf)
    cel(ws2,row,6,d.get('frequency',''),rf); cel(ws2,row,7,d.get('review_date_rule',''),rf)
    cel(ws2,row,8,d.get('effective_date_rule',''),rf); cel(ws2,row,9,d.get('june_2026_rebalancing',''),rf)
    ws2.row_dimensions[row].height = 45
ws2.freeze_panes = 'A3'

# ── Sheet 3: KRX Rule Engine ──
ws3 = wb.create_sheet('KRX_Rule_Engine_파라미터')
ws3.merge_cells('A1:N1')
t3 = ws3.cell(1,1,'KRX 지수 8단계 Rule Engine 파라미터')
t3.font = Font(bold=True,size=13,color='FFFFFF'); t3.fill = PatternFill('solid',fgColor='7030A0'); t3.alignment = Alignment(horizontal='center',vertical='center')
ws3.row_dimensions[1].height = 28
hdrs3 = ['No','파일명','지수명','섹터수','섹터분류','cap_threshold','liq_pct','keep_buffer','new_buffer','tertiary_mode','target_n','large_cap_special','ceiling_pct','June2026']
for c,h in enumerate(hdrs3,1): hdr(ws3,2,c,h)
ws3.row_dimensions[2].height = 40
for i,w in enumerate([5,32,30,8,12,13,8,12,10,14,9,15,11,8],1): ws3.column_dimensions[get_column_letter(i)].width = w
krx = [d for d in ALL_INDEX_DATA if d.get('provider_type')=='KRX' and d.get('index_type')=='주식']
for idx,d in enumerate(sorted(krx,key=lambda x:x.get('file','')),1):
    row=idx+2; jv=str(d.get('june_2026_rebalancing','')); rf=june_fill(jv)
    cel(ws3,row,1,idx,rf); cel(ws3,row,2,d.get('file',''),rf); cel(ws3,row,3,d.get('index_name_ko',''),rf)
    cel(ws3,row,4,d.get('sector_count'),rf); cel(ws3,row,5,d.get('sector_map_type',''),rf)
    cel(ws3,row,6,d.get('cap_threshold'),rf); cel(ws3,row,7,d.get('liq_pct'),rf)
    cel(ws3,row,8,d.get('keep_buffer'),rf); cel(ws3,row,9,d.get('new_buffer'),rf)
    cel(ws3,row,10,d.get('tertiary_mode',''),rf); cel(ws3,row,11,d.get('target_n'),rf)
    cel(ws3,row,12,d.get('large_cap_special_rank'),rf); cel(ws3,row,13,d.get('ceiling_pct'),rf)
    js='Y' if jv.startswith('Y') else('N' if jv.startswith('N') else '-')
    c=ws3.cell(row,14,js); c.fill=rf; c.font=Font(bold=True); c.alignment=Alignment(horizontal='center',vertical='center'); c.border=thin_border
    ws3.row_dimensions[row].height = 35
ws3.freeze_panes = 'A3'

# ── Sheet 4: 전체 기본정보 ──
ws4 = wb.create_sheet('전체_지수_기본정보')
ws4.merge_cells('A1:K1')
t4 = ws4.cell(1,1,'전체 100개 지수 방법론 기본 정보 (v2)')
t4.font = Font(bold=True,size=13,color='FFFFFF'); t4.fill = PatternFill('solid',fgColor='1F4E79'); t4.alignment = Alignment(horizontal='center',vertical='center')
ws4.row_dimensions[1].height = 28
hdrs4 = ['No','파일명','지수명(한글)','지수명(영문)','지수사','유형','지수유형','주기','심사기준일','변경적용일','June2026']
for c,h in enumerate(hdrs4,1): hdr(ws4,2,c,h)
ws4.row_dimensions[2].height = 35
for i,w in enumerate([5,28,30,35,18,12,10,18,35,40,8],1): ws4.column_dimensions[get_column_letter(i)].width = w
for idx,d in enumerate(ALL_INDEX_DATA,1):
    row=idx+2; jv=str(d.get('june_2026_rebalancing','')); rf=june_fill(jv) if jv.startswith('Y') else (None if jv.startswith('N') else na_fill)
    cel(ws4,row,1,idx,rf); cel(ws4,row,2,d.get('file',''),rf); cel(ws4,row,3,d.get('index_name_ko',''),rf)
    cel(ws4,row,4,d.get('index_name_en',''),rf); cel(ws4,row,5,d.get('provider',''),rf)
    cel(ws4,row,6,d.get('provider_type',''),rf); cel(ws4,row,7,d.get('index_type',''),rf)
    cel(ws4,row,8,d.get('frequency',''),rf); cel(ws4,row,9,d.get('review_date_rule',''),rf)
    cel(ws4,row,10,d.get('effective_date_rule',''),rf)
    js='Y' if jv.startswith('Y') else('N' if jv.startswith('N') else '-')
    c=ws4.cell(row,11,js); c.fill=june_fill(jv); c.font=Font(bold=True); c.alignment=Alignment(horizontal='center',vertical='center'); c.border=thin_border
    ws4.row_dimensions[row].height = 40
ws4.freeze_panes = 'A3'

# ── Sheet 5: 통계 ──
ws5 = wb.create_sheet('제공사별_통계')
ws5.merge_cells('A1:E1')
t5 = ws5.cell(1,1,'제공사별 2026년 6월 리밸런싱 통계')
t5.font = Font(bold=True,size=13,color='FFFFFF'); t5.fill = PatternFill('solid',fgColor='1F4E79'); t5.alignment = Alignment(horizontal='center',vertical='center')
ws5.row_dimensions[1].height = 28
for c,h in enumerate(['제공사 유형','Y (리밸런싱)','N (없음)','해당없음','합계'],1): hdr(ws5,2,c,h)
ws5.row_dimensions[2].height = 35
for i,w in enumerate([20,15,12,12,10],1): ws5.column_dimensions[get_column_letter(i)].width = w
summary_stats = defaultdict(lambda:{'Y':0,'N':0,'NA':0})
for d in ALL_INDEX_DATA:
    pt = d.get('provider_type','기타'); jv = str(d.get('june_2026_rebalancing',''))
    if jv.startswith('Y'): summary_stats[pt]['Y']+=1
    elif jv.startswith('N'): summary_stats[pt]['N']+=1
    else: summary_stats[pt]['NA']+=1
fills5 = [PatternFill('solid',fgColor=c) for c in ['EBF3FB','FFFFFF','EBF3FB','FFFFFF','EBF3FB','FFFFFF']]
for idx,(pt,counts) in enumerate(sorted(summary_stats.items()),1):
    row=idx+2; rf=fills5[idx%2]
    cel(ws5,row,1,pt,rf); cel(ws5,row,2,counts['Y'],yes_fill if counts['Y'] else rf)
    cel(ws5,row,3,counts['N'],no_fill if counts['N'] else rf)
    cel(ws5,row,4,counts['NA'],na_fill if counts['NA'] else rf)
    cel(ws5,row,5,counts['Y']+counts['N']+counts['NA'],rf)
    ws5.row_dimensions[row].height = 30
# Total row
tr = len(summary_stats)+3
total_y=sum(v['Y'] for v in summary_stats.values()); total_n=sum(v['N'] for v in summary_stats.values()); total_na=sum(v['NA'] for v in summary_stats.values())
hdr(ws5,tr,1,'합계'); hdr(ws5,tr,2,total_y); hdr(ws5,tr,3,total_n); hdr(ws5,tr,4,total_na); hdr(ws5,tr,5,total_y+total_n+total_na)
ws5.row_dimensions[tr].height = 30

output_path = 'index_methodology_analysis_v2_2026.xlsx'
wb.save(output_path)
print(f'Excel 저장 완료: {output_path}')
print(f'시트: June2026_리밸런싱_요약, June2026_Y_리밸런싱, KRX_Rule_Engine_파라미터, 전체_지수_기본정보, 제공사별_통계')
print(f'전체={len(ALL_INDEX_DATA)}, Y={total_y}, N={total_n}, 해당없음={total_na}')

## 8. MotherDuck 연결 및 dim_etf 매칭

> **주의**: 이 섹션은 `extensions.duckdb.org`에 접근 가능한 환경(로컬 PC 등)에서만 실행됩니다.  
> Claude Code 원격 환경에서는 네트워크 정책으로 인해 DuckDB 확장 다운로드가 차단됩니다.

**로컬 실행 방법:**
```bash
pip install duckdb pandas
```

**MotherDuck 토큰**: `.claude/settings.json`의 `MOTHERDUCK_TOKEN` 환경변수 참조

In [ ]:
# MotherDuck 연결 (로컬 환경 필요)
# pip install duckdb

MOTHERDUCK_TOKEN = os.environ.get('MOTHERDUCK_TOKEN', '')

try:
    import duckdb
    if not MOTHERDUCK_TOKEN:
        raise ValueError('MOTHERDUCK_TOKEN 환경변수를 설정하세요')
    
    con = duckdb.connect(f'md:?motherduck_token={MOTHERDUCK_TOKEN}')
    print('MotherDuck 연결 성공!')
    
    etf_index_df = con.execute('SELECT DISTINCT etf_index FROM dim_etf WHERE etf_index IS NOT NULL ORDER BY etf_index').df()
    print(f'etf_index 고유값: {len(etf_index_df)}개')
    print(etf_index_df.head(20))
    
except Exception as e:
    print(f'연결 실패: {e}')
    print('로컬 환경에서 실행하세요 (pip install duckdb)')

## 9. etf_index 매칭

In [ ]:
import re

def normalize(text):
    if not text: return ''
    text = str(text).lower()
    text = re.sub(r'[\s\-_·•]+', '', text)
    text = re.sub(r'[^가-힣a-z0-9]', '', text)
    return text

# etf_index_list를 MotherDuck에서 가져오거나 직접 입력
# etf_index_list = etf_index_df['etf_index'].tolist()  # MotherDuck 연결 후
etf_index_list = []  # 직접 입력 예시: ['KOSPI 200', 'KOSDAQ 150', ...]

if etf_index_list:
    results = []
    for etf_idx in etf_index_list:
        norm_etf = normalize(etf_idx)
        best_match = None; best_score = 0
        for d in ALL_INDEX_DATA:
            for name_field in ['index_name_ko', 'index_name_en']:
                nm = normalize(d.get(name_field,''))
                if nm and norm_etf in nm: best_score = max(best_score, len(norm_etf)/len(nm))
                if best_score > 0: best_match = d
        results.append({'etf_index': etf_idx, 'matched': best_match.get('index_name_ko','') if best_match else '매칭없음',
                        'june_2026': best_match.get('june_2026_rebalancing','') if best_match else '-'})
    
    for r in results:
        print(f'{r["etf_index"]:<35} → {r["matched"]:<30} | June2026: {r["june_2026"]}')
else:
    print('etf_index_list를 채운 후 실행하세요 (MotherDuck 연결 또는 직접 입력)')

## 10. 핵심 요약

### 2026년 6월 리밸런싱 현황 (v2 분석)

| 구분 | 수량 |
|------|------|
| 전체 지수 | 100개 |
| **6월 리밸런싱 Y** | **57개** |
| 6월 리밸런싱 N | 20개 |
| 해당없음/파생/옵션 | 23개 |

### 제공사별 Y 비율
| 제공사 | Y | 전체 |
|--------|---|------|
| KRX | ~22 | 30 |
| FnGuide계열 | ~22 | 38 |
| iSelect | ~7 | 14 |
| KEDI | ~4 | 5 |
| MSCI | 0 | 8 |
| 기타 | ~2 | 5 |

### KRX 표준 Rule Engine 파라미터
- cap_threshold: **85%** (대부분), 특수 지수는 다름
- liq_pct: **85%**
- keep_buffer: **110%** (편입유지 버퍼)
- new_buffer: **90%** (신규편입 버퍼)
- tertiary_mode: **global** (기본값)
- large_cap_special_rank: **50위** (시총 상위 50 자동편입)